# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
# import os
# os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"

# # Install uv (curl works on DataHub; falls back to wget if needed)
# !curl -LsSf https://astral.sh/uv/install.sh | sh || wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment (use full path in case ~/.local/bin still isn't picked up)
# !~/.local/bin/uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment. 

In [2]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [3]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())

Sat May  9 19:50:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:DA:00.0 Off |                  N/A |
|  0%   26C    P8             19W /  250W |       1MiB /  11264MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

INFO 05-09 22:48:18 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 05-09 22:48:21 [__init__.py:239] Automatically detected platform cuda.


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [4]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [5]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [6]:
# !.venv/bin/python -m pip uninstall -y \
#     torch torchvision torchaudio \
#     vllm flashinfer-python flashinfer-cubin tilelang triton \
#     nvidia-cublas nvidia-cuda-cupti nvidia-cuda-nvrtc nvidia-cuda-runtime \
#     nvidia-cudnn-cu13 nvidia-cudnn-frontend nvidia-cufft nvidia-cufile \
#     nvidia-curand nvidia-cusolver nvidia-cusparse nvidia-cusparselt-cu13 \
#     nvidia-nccl-cu13 nvidia-nvjitlink nvidia-nvshmem-cu13 nvidia-nvtx \
#     nvidia-cutlass-dsl nvidia-cutlass-dsl-libs-base \
#     cuda-toolkit cuda-bindings cuda-python cuda-pathfinder cuda-tile \
#     apache-tvm-ffi quack-kernels

# !.venv/bin/python -m pip install "vllm==0.8.5.post1"

In [7]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# tokenizer.pad_token = tokenizer.eos_token

# llm = LLM(
#     model=MODEL_ID,
#     quantization="bitsandbytes",
#     load_format="bitsandbytes",
#     enable_prefix_caching=False,
#     gpu_memory_utilization=0.50,
#     max_model_len=16384,
#     trust_remote_code=True,
#     max_num_seqs=256,
#     max_num_batched_tokens=32768,
# )

# sampling_params = SamplingParams(
#     max_tokens=MAX_TOKENS,
#     temperature=0.6,
#     top_p=0.95,
#     top_k=20,
#     min_p=0.0,
#     presence_penalty=0.0,
#     repetition_penalty=1.0,
# )

# print("Model loaded.")

## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [8]:
# pip install accelerate

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [10]:
# # Build prompts for first 5 entries
# prompts = []
# for item in data[:5]:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)

# # Generate
# print(f"Generating responses for {len(prompts)} questions...")
# outputs = llm.generate(prompts, sampling_params=sampling_params)

# responses = [out.outputs[0].text.strip() for out in outputs]

# # Preview first 3
# for i in range(min(3, len(responses))):
#     print(f"\n── Response {i} (id={data[i].get('id')}) ──")
#     print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

### Generate with Transformers (for Datahub)

In [11]:
import gc

GEN_MAX_NEW_TOKENS = 4096
SUBSET = data[:5]

prompts = []
for item in SUBSET:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

print(f"Generating responses for {len(prompts)} questions (one at a time)...")

responses = []
for i, prompt_text in enumerate(tqdm(prompts, desc="Generating")):
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=8192,
    ).to(llm.device)

    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            temperature=0.6,
            top_p=0.95,
            top_k=20,
            repetition_penalty=1.0,
            do_sample=True,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

    del inputs, output_ids, new_tokens
    gc.collect()
    torch.cuda.empty_cache()

for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={SUBSET[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 5 questions (one at a time)...


Generating:   0%|          | 0/5 [00:10<?, ?it/s]


KeyboardInterrupt: 

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [16]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   0%|          | 0/1126 [00:00<?, ?it/s]

Scoring complete. 0 results.


## 8. Summary

Print accuracy broken down by question type.

In [9]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    1 /    2  (50.00%)
  Free-form  :    2 /    3  (66.67%)
  Overall    :    3 /    5  (60.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [10]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 5 records to results/starter_results.jsonl


## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!

In [12]:
import random

PROMPT_VARIANTS = {
    "A_baseline": {
        "math": SYSTEM_PROMPT_MATH,
        "mcq":  SYSTEM_PROMPT_MCQ,
    },
    "B_verify": {
        "math": (
            "You are an expert mathematician. Solve the problem step-by-step. "
            "After deriving an answer, verify it using an independent method "
            "(substitution, dimensional check, or alternative derivation). "
            "If verification fails, reconsider and correct your answer. "
            "Put ONLY your final answer inside \\boxed{}, with no surrounding text. "
            "For multi-part problems, separate parts by commas inside one box, e.g. \\boxed{3, 7}."
        ),
        "mcq": (
            "You are an expert mathematician. Read the problem and answer choices carefully. "
            "Eliminate clearly wrong options first, then verify the chosen option satisfies the problem. "
            "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
        ),
    },
    "C_strict_format": {
        "math": (
            "You are an expert mathematician. Solve concisely, showing only essential steps. "
            "Place ONLY the final numeric or symbolic answer inside \\boxed{}: no units, "
            "no surrounding words, no trailing punctuation. "
            "For multi-part problems, separate parts by commas inside one box: \\boxed{a, b}. "
            "Output the box only once, at the very end of your response."
        ),
        "mcq": (
            "You are an expert mathematician. Choose the single best option. "
            "Output exactly one character inside \\boxed{}: the letter A-J of your choice, "
            "e.g. \\boxed{C}. Output the box only once, at the very end."
        ),
    },
}


def build_prompt_variant(variant_id: str, question: str, options) -> tuple[str, str]:
    sys_pack = PROMPT_VARIANTS[variant_id]
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return sys_pack["mcq"], f"{question}\n\nOptions:\n{opts_text}"
    return sys_pack["math"], question


SUBSET_SIZE = 10  # 5 MCQ + 5 free-form. Bump up later for tighter estimates.
random.seed(42)

mcq_pool   = [it for it in data if it.get("options")]
free_pool  = [it for it in data if not it.get("options")]
n_mcq_want = SUBSET_SIZE // 2

subset = random.sample(mcq_pool, n_mcq_want) + random.sample(free_pool, SUBSET_SIZE - n_mcq_want)

print(f"Subset: {len(subset)} questions  "
      f"({sum(bool(it.get('options')) for it in subset)} MCQ, "
      f"{sum(not it.get('options') for it in subset)} free-form)")
print(f"Variants: {list(PROMPT_VARIANTS)}")

Subset: 10 questions  (5 MCQ, 5 free-form)
Variants: ['A_baseline', 'B_verify', 'C_strict_format']


In [14]:
import gc
from pathlib import Path

METHOD_MAX_NEW_TOKENS = 2048
CKPT_DIR = Path("results/method_comparison/_ckpt")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def _load_done(variant_id: str) -> dict[int, str]:
    """Read whatever is already saved for this variant; returns {item_id: response}."""
    path = CKPT_DIR / f"{variant_id}.jsonl"
    done = {}
    if path.exists():
        with path.open() as f:
            for line in f:
                rec = json.loads(line)
                done[rec["id"]] = rec["response"]
    return done

variant_responses = {v: [] for v in PROMPT_VARIANTS}

for variant_id in PROMPT_VARIANTS:
    ckpt_path = CKPT_DIR / f"{variant_id}.jsonl"
    done = _load_done(variant_id)
    if done:
        print(f"\n=== Variant: {variant_id} (resuming, {len(done)}/{len(subset)} already done) ===")
    else:
        print(f"\n=== Variant: {variant_id} ===")

    with ckpt_path.open("a") as out_f:
        for item in tqdm(subset, desc=variant_id):
            qid = item.get("id")

            if qid in done:
                variant_responses[variant_id].append(done[qid])
                continue

            sys_p, usr_p = build_prompt_variant(variant_id, item["question"], item.get("options"))
            prompt_text = tokenizer.apply_chat_template(
                [{"role": "system", "content": sys_p},
                 {"role": "user",   "content": usr_p}],
                tokenize=False,
                add_generation_prompt=True,
            )
            inputs = tokenizer(
                prompt_text, return_tensors="pt", truncation=True, max_length=8192,
            ).to(llm.device)

            with torch.no_grad():
                output_ids = llm.generate(
                    **inputs,
                    max_new_tokens=METHOD_MAX_NEW_TOKENS,
                    temperature=0.6, top_p=0.95, top_k=20,
                    repetition_penalty=1.0, do_sample=True,
                    use_cache=True, pad_token_id=tokenizer.eos_token_id,
                )

            new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
            response  = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            variant_responses[variant_id].append(response)

            out_f.write(json.dumps({"id": qid, "response": response}) + "\n")
            out_f.flush()

            del inputs, output_ids, new_tokens
            gc.collect()
            torch.cuda.empty_cache()

print("\nAll variants generated.")


=== Variant: A_baseline ===


A_baseline: 100%|██████████| 10/10 [25:44<00:00, 154.45s/it]



=== Variant: B_verify ===


B_verify: 100%|██████████| 10/10 [25:10<00:00, 151.06s/it]



=== Variant: C_strict_format ===


C_strict_format: 100%|██████████| 10/10 [22:07<00:00, 132.70s/it]


All variants generated.


In [17]:
def score_one(item, response):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]
    if is_mcq:
        return score_mcq(response, str(gold))
    gold_list = gold if isinstance(gold, list) else [gold]
    try:
        return judger.auto_judge(
            pred=response,
            gold=gold_list,
            options=[[]] * len(gold_list),
        )
    except Exception:
        return False


variant_correct = {v: [score_one(it, r) for it, r in zip(subset, variant_responses[v])]
                   for v in PROMPT_VARIANTS}

n_mcq  = sum(bool(it.get("options")) for it in subset)
n_free = len(subset) - n_mcq

print(f"\n{'Variant':<18} {'Overall':>8} {'MCQ':>8} {'FreeForm':>10}")
print("-" * 46)
for v, correct in variant_correct.items():
    acc_all  = sum(correct) / len(correct)
    acc_mcq  = sum(c for c, it in zip(correct, subset) if it.get("options")) / max(n_mcq, 1)
    acc_free = sum(c for c, it in zip(correct, subset) if not it.get("options")) / max(n_free, 1)
    print(f"{v:<18} {acc_all:>7.1%} {acc_mcq:>7.1%} {acc_free:>9.1%}")

print("\n─── Disagreements (questions where variants score differently) ───")
shown = 0
for i, item in enumerate(subset):
    flags = [variant_correct[v][i] for v in PROMPT_VARIANTS]
    if len(set(flags)) > 1:
        shown += 1
        print(f"\n[id={item.get('id')}]  type={'MCQ' if item.get('options') else 'free'}  "
              f"gold={item['answer']!r}")
        q_preview = item["question"].replace("\n", " ")[:160]
        print(f"  Q: {q_preview}{'...' if len(item['question']) > 160 else ''}")
        for v in PROMPT_VARIANTS:
            mark    = "✓" if variant_correct[v][i] else "✗"
            tail    = variant_responses[v][i][-220:].replace("\n", " ")
            print(f"  {mark} {v:<16} ...{tail}")

if shown == 0:
    print("  (No disagreements on this subset — all variants agreed on every question.)")


Variant             Overall      MCQ   FreeForm
----------------------------------------------
A_baseline           30.0%   20.0%     40.0%
B_verify             10.0%    0.0%     20.0%
C_strict_format      30.0%   20.0%     40.0%

─── Disagreements (questions where variants score differently) ───

[id=974]  type=MCQ  gold='G'
  Q: Consider a triangle $\triangle HGU$. The centroid is denoted by $B$, while $X$ represents the incenter. The angles at vertices $H$ and $G$ are labeled as $N$ an...
  ✓ A_baseline       ...it, in standard triangle notation, \( a \) is the length of \( BC \) in \( \triangle ABC \), so opposite \( A \). So in \( \triangle HGU \), let's denote:  - \( h = |GU| \) (length of side opposite \( H \)) - \( g = |HU|
  ✗ B_verify         ...gle \( \triangle ABC \), with \( A, B, C \) vertices, the incenter is \( \frac{aA + bB + cC}{a + b + c} \) where \( a = |BC| \), \( b = |AC| \), \( c = |AB| \). So here, \( A = H \), \( B = G \), \( C = U \), so:  - \( a
  ✗ C_strict

In [18]:
from pathlib import Path

METHOD_DIR = Path("results/method_comparison")
METHOD_DIR.mkdir(parents=True, exist_ok=True)

for v in PROMPT_VARIANTS:
    out_path = METHOD_DIR / f"{v}.jsonl"
    with out_path.open("w") as f:
        for item, resp, ok in zip(subset, variant_responses[v], variant_correct[v]):
            f.write(json.dumps({
                "id":       item.get("id"),
                "is_mcq":   bool(item.get("options")),
                "gold":     item["answer"],
                "response": resp,
                "correct":  bool(ok),
            }) + "\n")
    print(f"Saved {out_path}  ({len(subset)} rows)")

summary = {
    "subset_size":            len(subset),
    "n_mcq":                  n_mcq,
    "n_free":                 n_free,
    "max_new_tokens":          METHOD_MAX_NEW_TOKENS,
    "seed":                   42,
    "variants": {
        v: {
            "overall_acc":  sum(variant_correct[v]) / len(subset),
            "mcq_acc":      sum(c for c, it in zip(variant_correct[v], subset) if it.get("options")) / max(n_mcq, 1),
            "freeform_acc": sum(c for c, it in zip(variant_correct[v], subset) if not it.get("options")) / max(n_free, 1),
        }
        for v in PROMPT_VARIANTS
    },
}
with (METHOD_DIR / "summary.json").open("w") as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved {METHOD_DIR / 'summary.json'}")
print(json.dumps(summary, indent=2))

Saved results/method_comparison/A_baseline.jsonl  (10 rows)
Saved results/method_comparison/B_verify.jsonl  (10 rows)
Saved results/method_comparison/C_strict_format.jsonl  (10 rows)

Saved results/method_comparison/summary.json
{
  "subset_size": 10,
  "n_mcq": 5,
  "n_free": 5,
  "max_new_tokens": 2048,
  "seed": 42,
  "variants": {
    "A_baseline": {
      "overall_acc": 0.3,
      "mcq_acc": 0.2,
      "freeform_acc": 0.4
    },
    "B_verify": {
      "overall_acc": 0.1,
      "mcq_acc": 0.0,
      "freeform_acc": 0.2
    },
    "C_strict_format": {
      "overall_acc": 0.3,
      "mcq_acc": 0.2,
      "freeform_acc": 0.4
    }
  }
}
